# PGT Bounce Gravitational Wave Spectrum Solver

**Branch M: Precision characterization of the GW spectrum from a lower-scale PGT spin-torsion bounce**

Date: 2026-03-16

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings('ignore')

## 1. Background Functions

PGT bounce with $\rho_{\rm crit} = m_T^2 M_{\rm Pl}^2$.

Scale factor: $a(t) = a_b (1 + 4\alpha^2 t^2)^{1/4}$ with $\alpha^2 = (8\pi/3) m_T^2$.

We work in units where $\alpha = 1$ (dimensionless time $\tau = \alpha t$),
and $a_b = 1$ (dimensionless scale factor $\tilde{a} = a/a_b$).

The bounce scale is then $k_b = \alpha a_b = 1$ in these units.
Physical wavenumber $\kappa = k / k_b$.

In [ ]:
def a_tau(tau):
    """Scale factor a(tau) / a_b in dimensionless units."""
    return (1 + 4 * tau**2)**0.25

def H_tau(tau):
    """Hubble parameter H(tau) * (1/alpha) in dimensionless units."""
    return 2 * tau / (1 + 4 * tau**2)

def Hdot_tau(tau):
    """dH/dtau in dimensionless units."""
    return 2 * (1 - 4 * tau**2) / (1 + 4 * tau**2)**2

def adotdot_over_a(tau):
    """(d^2a/dtau^2)/a = Hdot + H^2 in dimensionless units."""
    return Hdot_tau(tau) + H_tau(tau)**2

# Conformal time by numerical integration
def eta_of_tau(tau_array):
    """Compute conformal time eta for array of cosmic times tau.
    eta = integral of dtau/a(tau)."""
    from scipy.integrate import cumulative_trapezoid
    dtau = np.diff(tau_array)
    integrand = 1.0 / a_tau(tau_array)
    eta_vals = np.zeros_like(tau_array)
    eta_vals[1:] = cumulative_trapezoid(integrand, tau_array)
    return eta_vals

## 2. Effective Potential

The tensor mode equation in conformal time:

$$\mu_\kappa'' + (\kappa^2 - \tilde{V}(\eta)) \mu_\kappa = 0$$

where $\tilde{V} = a''/a = a^2 \times (\ddot{a}/a + H^2)$.

In cosmic time:

$$\tilde{V}(\tau) = a^2(\tau) \times [\dot{H}(\tau) + 2H^2(\tau)]$$

Wait — the correct relation is $a''/a = a^2(\ddot{a}/a + H^2) = a^2(\dot{H} + 2H^2)$.

Actually: $a' = \dot{a} \cdot a$, $a'' = (\ddot{a} a + \dot{a}^2) a = a^2(\ddot{a}/a + H^2)$.
So $a''/a = a^2(\ddot{a}/a + H^2) = a^2(\dot{H} + 2H^2)$.

In [ ]:
def V_eff_cosmic(tau):
    """Effective potential a''/a computed from cosmic time quantities.
    V = a^2 * (Hdot + 2H^2) in dimensionless units."""
    a = a_tau(tau)
    H = H_tau(tau)
    Hd = Hdot_tau(tau)
    return a**2 * (Hd + 2 * H**2)

# Verify: at tau=0, V = 1^2 * (2 + 0) = 2  (matches analytic result)
print(f"V_eff at bounce (tau=0): {V_eff_cosmic(0):.4f} (expect 2.0000)")

# Plot the potential
tau_plot = np.linspace(-5, 5, 1000)
V_plot = np.array([V_eff_cosmic(t) for t in tau_plot])

print(f"V_eff peak: {np.max(V_plot):.4f}")
print(f"V_eff at tau=1: {V_eff_cosmic(1):.6f}")
print(f"V_eff at tau=3: {V_eff_cosmic(3):.6f}")

## 3. Mode Evolution in Cosmic Time

We solve the tensor equation in cosmic time (easier numerics for the bounce).

The equation for $h_\kappa(\tau)$:

$$\ddot{h}_\kappa + 3H \dot{h}_\kappa + (\kappa^2/a^2) h_\kappa = 0$$

Wait — for tensors in radiation, the friction term is $2H$ not $3H$.
Actually, for transverse-traceless tensor perturbations:

$$\ddot{h}_\kappa + 3H \dot{h}_\kappa + (k^2/a^2) h_\kappa = 0$$

This is the standard GW propagation equation.

In [ ]:
def tensor_rhs(tau, y, kappa):
    """RHS for tensor mode equation.
    y = [h, hdot]
    h'' + 3H h' + (kappa^2/a^2) h = 0
    """
    h, hdot = y
    H = H_tau(tau)
    a = a_tau(tau)
    h_ddot = -3 * H * hdot - (kappa**2 / a**2) * h
    return [hdot, h_ddot]

def solve_tensor_mode(kappa, tau_i=-50, tau_f=50, N=100000):
    """Solve the tensor mode equation for wavenumber kappa.
    
    Initial condition: incoming positive-frequency WKB mode
    in the contracting phase (tau -> -infty).
    
    h ~ (1/a) * exp(-i * kappa * eta) / sqrt(2*kappa)
    """
    # At early time (tau_i << 0), we're in the radiation era:
    # a ~ a_b * (2|tau_i|)^{1/2}, H ~ 1/(2*tau_i)
    # WKB solution: h = A * exp(-i*kappa*eta) / (a * sqrt(2*kappa))
    # Need eta at tau_i
    a_i = a_tau(tau_i)
    
    # In the radiation era (|tau| >> 1): eta ~ 2*sqrt(|tau|) / a_b
    # More precisely: eta = integral dt/a from 0 to tau_i
    # For large |tau|: a ~ (2|tau|)^{1/2}, eta ~ 2*sqrt(|tau|)
    # But we need the sign convention. For tau_i < 0 (contraction):
    # eta is negative and increasing toward 0.
    
    # Physical frequency in dimensionless units: omega = kappa/a
    omega_i = kappa / a_i
    
    # WKB initial conditions (plane wave, normalized)
    # h = cos(omega * delta_eta) / (a * sqrt(2*kappa)) as real part
    # We use two linearly independent real solutions:
    # Solution 1: h = cos(phi) / (a * sqrt(2*kappa))
    # Solution 2: h = sin(phi) / (a * sqrt(2*kappa))
    # where phi = kappa * eta + phase
    
    # For Bogoliubov coefficients, solve with two initial conditions
    # and extract alpha, beta from the late-time WKB matching.
    
    # IC 1: cosine mode
    h1_0 = 1.0 / (a_i * np.sqrt(2 * kappa))
    # dh/dtau = -(H/a)*cos(phi)/(sqrt(2k)) - (omega/a)*sin(phi)/(sqrt(2k))
    # At the initial time, phi = 0 convention:
    h1dot_0 = -H_tau(tau_i) * h1_0  # cosine part: hdot = -H*h (frozen amplitude)
    
    # IC 2: sine mode  
    h2_0 = 0.0
    h2dot_0 = omega_i / (a_i * np.sqrt(2 * kappa))  # sine mode velocity
    
    tau_span = (tau_i, tau_f)
    tau_eval = np.linspace(tau_i, tau_f, N)
    
    sol1 = solve_ivp(tensor_rhs, tau_span, [h1_0, h1dot_0],
                     args=(kappa,), t_eval=tau_eval,
                     method='DOP853', rtol=1e-12, atol=1e-14)
    
    sol2 = solve_ivp(tensor_rhs, tau_span, [h2_0, h2dot_0],
                     args=(kappa,), t_eval=tau_eval,
                     method='DOP853', rtol=1e-12, atol=1e-14)
    
    return sol1, sol2

print("Tensor mode solver defined.")

## 4. Bogoliubov Coefficient Extraction

At late times ($\tau \gg 1$), the mode is in the WKB regime.
We extract $|\beta_\kappa|^2$ by comparing the amplitude of the
outgoing mode to the vacuum normalization.

In [ ]:
def extract_bogoliubov(kappa, tau_i=-80, tau_f=80):
    """Extract |beta_k|^2 for wavenumber kappa.
    
    Strategy: solve the canonical mode equation mu'' + (k^2 - V) mu = 0
    with positive-frequency initial conditions, then extract the
    negative-frequency component at late times.
    """
    # Use the canonical variable mu = a * h
    # mu'' + (kappa^2 - a''/a) mu = 0
    
    # We work directly in cosmic time with variable mu:
    # d^2(mu)/deta^2 + (k^2 - V) mu = 0
    # Converting: deta = dtau/a, so d/deta = a * d/dtau
    # d^2/deta^2 = a^2 d^2/dtau^2 + a*adot d/dtau
    
    # Actually, let's just solve in cosmic time for h and compute mu = a*h.
    # Then at late times, mu oscillates as:
    # mu ~ alpha * e^{-i k eta} / sqrt(2k) + beta * e^{+i k eta} / sqrt(2k)
    
    # Simpler approach: use the transfer matrix method.
    # Solve two independent solutions, match to WKB at both ends.
    
    a_i = a_tau(tau_i)
    a_f = a_tau(tau_f)
    omega_i = kappa / a_i
    omega_f = kappa / a_f
    
    # Solve h equation with IC: h(tau_i) = 1, hdot(tau_i) = 0
    sol_a = solve_ivp(tensor_rhs, (tau_i, tau_f), [1.0, 0.0],
                      args=(kappa,), method='DOP853',
                      rtol=1e-12, atol=1e-15,
                      dense_output=True)
    
    # Solve h equation with IC: h(tau_i) = 0, hdot(tau_i) = 1
    sol_b = solve_ivp(tensor_rhs, (tau_i, tau_f), [0.0, 1.0],
                      args=(kappa,), method='DOP853',
                      rtol=1e-12, atol=1e-15,
                      dense_output=True)
    
    # Get values at tau_f
    h_a_f = sol_a.sol(tau_f)[0]
    hdot_a_f = sol_a.sol(tau_f)[1]
    h_b_f = sol_b.sol(tau_f)[0]
    hdot_b_f = sol_b.sol(tau_f)[1]
    
    # Transfer matrix: [h(tau_f), hdot(tau_f)] = M [h(tau_i), hdot(tau_i)]
    # M = [[h_a_f, h_b_f], [hdot_a_f, hdot_b_f]]
    
    # WKB at tau_i (contracting phase, H < 0):
    # Positive frequency: h ~ exp(-i*omega*eta)/(a*sqrt(2k))
    # In cosmic time: h ~ exp(-i*phi(tau)) / (a*sqrt(2k))
    # where dphi/dtau = omega = k/a
    
    # At late times, the mode decomposes as:
    # h = alpha * h_+ + beta * h_-  (positive and negative frequency)
    
    # For a simpler approach: use the constancy of the Wronskian.
    # The produced particle number is related to the ratio of 
    # the mode amplitude at late times to the vacuum normalization.
    
    # Use energy method: compare the energy in the mode at tau_f
    # to the vacuum energy.
    
    # h_k oscillates at late times as:
    # h = A cos(omega_f * delta_eta_f + phi_f) / a_f
    # where omega_f = k/a_f
    
    # The vacuum amplitude is 1/(a_f * sqrt(2k)).
    # The actual amplitude includes 1 + 2|beta|^2 particles.
    
    # Amplitude^2 of oscillation = h^2 + (hdot/omega)^2 (time-averaged)
    # For WKB: this is |A|^2 / a_f^2
    
    # Use the positive-frequency initial condition:
    # h(tau_i) = 1/(a_i * sqrt(2*kappa))
    # hdot(tau_i) = -H_i * h + i*omega_i * h  (positive frequency)
    # For real solutions, take two ICs and combine.
    
    # Cosine IC at input:
    h_i_cos = 1.0 / (a_i * np.sqrt(2 * kappa))
    hdot_i_cos = -H_tau(tau_i) * h_i_cos
    
    # Sine IC at input:
    h_i_sin = 0.0
    hdot_i_sin = omega_i * h_i_cos  # = omega_i / (a_i * sqrt(2k))
    
    # Evolve using transfer matrix
    h_cos_f = h_a_f * h_i_cos + h_b_f * hdot_i_cos
    hdot_cos_f = hdot_a_f * h_i_cos + hdot_b_f * hdot_i_cos
    
    h_sin_f = h_a_f * h_i_sin + h_b_f * hdot_i_sin
    hdot_sin_f = hdot_a_f * h_i_sin + hdot_b_f * hdot_i_sin
    
    # At tau_f, the WKB mode oscillates.
    # The complex positive-frequency solution is:
    # h_+ = (h_cos + i*h_sin)
    # At output, decompose into positive and negative frequency:
    # h_+ = alpha * h_+_out + beta * h_-_out
    
    # The time-averaged amplitude^2 of the cos solution:
    H_f = H_tau(tau_f)
    
    # Adiabatic invariant: E_k = (1/2)(hdot + H*h)^2 + (1/2)(k/a)^2 h^2
    # For the vacuum: E_k = omega_f / (2 a_f^2) * (1 + 2|beta|^2)
    
    # Compute adiabatic invariant for each solution
    E_cos = 0.5 * (hdot_cos_f + H_f * h_cos_f)**2 + 0.5 * omega_f**2 * h_cos_f**2
    E_sin = 0.5 * (hdot_sin_f + H_f * h_sin_f)**2 + 0.5 * omega_f**2 * h_sin_f**2
    
    # The positive-frequency combination h_+ = h_cos + i*h_sin
    # has energy E_+ = E_cos + E_sin
    E_total = E_cos + E_sin
    
    # Vacuum energy: E_vac = omega_f / (2 * a_f^2 * 2*kappa) * 2 = omega_f^2 / (2*kappa*a_f^2)
    # Hmm, let me be more careful.
    
    # For a normalized positive-frequency WKB mode at tau_f:
    # h = exp(-i*int omega dt) / (a_f * sqrt(2*kappa))
    # |h|^2 = 1 / (a_f^2 * 2*kappa)
    # |hdot + H*h|^2 = omega_f^2 / (a_f^2 * 2*kappa)
    # E_vac = omega_f^2 / (a_f^2 * 2*kappa) per complex dof
    # But we split into cos+sin, each real, so:
    E_vac = omega_f**2 / (a_f**2 * 2 * kappa)
    
    # The ratio gives 1 + 2|beta|^2:
    ratio = E_total / E_vac
    beta_sq = (ratio - 1) / 2
    
    # Protect against numerical noise making beta_sq slightly negative
    beta_sq = max(beta_sq, 0)
    
    return beta_sq

print("Bogoliubov extraction defined.")

## 5. Compute |β_κ|² vs κ

In [ ]:
# Scan over kappa = k/k_b from 0.01 to 10
kappas = np.concatenate([
    np.logspace(-2, -0.5, 20),  # 0.01 to 0.316
    np.linspace(0.35, 3.0, 30),  # 0.35 to 3.0
    np.logspace(np.log10(3.5), 1.0, 15)  # 3.5 to 10
])

beta_sq = np.zeros_like(kappas)

# Adjust integration range based on kappa
for i, kap in enumerate(kappas):
    # Need tau_range >> 1/kappa for WKB to be valid
    tau_range = max(80, 20 / kap)
    try:
        beta_sq[i] = extract_bogoliubov(kap, tau_i=-tau_range, tau_f=tau_range)
    except Exception as e:
        print(f"Failed at kappa={kap:.4f}: {e}")
        beta_sq[i] = np.nan

# Report results
print("\nkappa\t\t|beta|^2")
print("-" * 40)
for k, b in zip(kappas, beta_sq):
    if not np.isnan(b):
        print(f"{k:.4f}\t\t{b:.6e}")

## 6. Compute Ω_GW(f)

$$\Omega_{\rm GW}(f) h^2 \approx 1.6 \times 10^{-5} \times \left(\frac{m_T}{M_{\rm Pl}}\right)^2 \times |\beta_\kappa|^2$$

where $\kappa = f/f_b$.

In [ ]:
def omega_gw_spectrum(m_T_GeV, kappas, beta_sq):
    """Compute Omega_GW h^2 for given m_T and Bogoliubov coefficients.
    
    Parameters:
        m_T_GeV: torsion mass in GeV
        kappas: array of k/k_b values
        beta_sq: array of |beta_k|^2 values
    
    Returns:
        f_Hz: array of frequencies in Hz
        Omega_h2: array of Omega_GW h^2 values
    """
    M_Pl_GeV = 1.22e19  # Planck mass in GeV
    
    # Bounce frequency
    f_b = 2.6e10 * np.sqrt(m_T_GeV / M_Pl_GeV)  # Hz
    
    # Physical frequencies
    f_Hz = kappas * f_b
    
    # Omega_GW h^2
    prefactor = 1.6e-5 * (m_T_GeV / M_Pl_GeV)**2
    Omega_h2 = prefactor * beta_sq
    
    return f_Hz, Omega_h2

# Compute for several m_T values
m_T_values = [1e7, 1e3, 1e-1, 1e-5, 1e-9]  # GeV

print("m_T (GeV)\tf_b (Hz)\t\tPeak Omega_GW h^2")
print("=" * 70)

for m_T in m_T_values:
    valid = ~np.isnan(beta_sq)
    f_Hz, Omega_h2 = omega_gw_spectrum(m_T, kappas[valid], beta_sq[valid])
    f_b = 2.6e10 * np.sqrt(m_T / 1.22e19)
    peak = np.max(Omega_h2) if len(Omega_h2) > 0 else 0
    print(f"{m_T:.1e}\t\t{f_b:.2e}\t\t{peak:.2e}")

## 7. Spectral Shape Analysis

In [ ]:
# Fit the spectral shape in different regimes
valid = ~np.isnan(beta_sq) & (beta_sq > 0)

if np.sum(valid) > 5:
    kap_v = kappas[valid]
    beta_v = beta_sq[valid]
    
    # Low-k regime (kappa < 0.3): fit power law
    low_k = kap_v < 0.3
    if np.sum(low_k) > 2:
        log_kap = np.log10(kap_v[low_k])
        log_beta = np.log10(beta_v[low_k])
        finite = np.isfinite(log_beta)
        if np.sum(finite) > 2:
            slope, intercept = np.polyfit(log_kap[finite], log_beta[finite], 1)
            print(f"Low-k power law: |beta|^2 ~ kappa^{slope:.2f}")
            print(f"  (expect ~0 for constant, ~2 for standard particle production)")
    
    # High-k regime (kappa > 3): fit exponential
    high_k = kap_v > 3.0
    if np.sum(high_k) > 2:
        kap_h = kap_v[high_k]
        beta_h = beta_v[high_k]
        pos = beta_h > 0
        if np.sum(pos) > 2:
            log_beta_h = np.log(beta_h[pos])  # natural log
            slope_h, intercept_h = np.polyfit(kap_h[pos], log_beta_h, 1)
            print(f"\nHigh-k exponential: |beta|^2 ~ exp({slope_h:.2f} * kappa)")
            print(f"  (expect ~ exp(-pi * kappa) = exp(-3.14 * kappa))")
    
    # Peak region
    peak_idx = np.argmax(beta_v)
    print(f"\nPeak: kappa = {kap_v[peak_idx]:.3f}, |beta|^2 = {beta_v[peak_idx]:.4e}")
    
    # FWHM estimate
    half_max = beta_v[peak_idx] / 2
    above_half = kap_v[beta_v > half_max]
    if len(above_half) > 1:
        fwhm = above_half[-1] - above_half[0]
        print(f"FWHM in kappa: {fwhm:.3f}")
else:
    print("Insufficient valid data points for shape analysis.")

## 8. Comparison with Detector Sensitivities

In [ ]:
# Detector sensitivity curves (approximate Omega_GW h^2)
detectors = {
    'LIGO O5': {'f_range': (10, 5000), 'Omega_h2': 1e-9},
    'ET': {'f_range': (1, 1e4), 'Omega_h2': 1e-13},
    'LISA': {'f_range': (1e-4, 0.1), 'Omega_h2': 1e-13},
    'DECIGO': {'f_range': (0.01, 100), 'Omega_h2': 1e-16},
    'NANOGrav': {'f_range': (1e-9, 1e-7), 'Omega_h2': 1e-10},
}

print("\nDetector reach vs PGT bounce spectrum")
print("=" * 80)
print(f"{'m_T (GeV)':<12} {'f_b (Hz)':<14} {'Peak Omega h2':<16} {'Best detector':<16} {'Gap'}")
print("-" * 80)

for m_T in [1e9, 1e7, 1e5, 1e3, 1e1, 1e-1, 1e-3, 1e-5, 1e-7, 1e-9]:
    f_b = 2.6e10 * np.sqrt(m_T / 1.22e19)
    Omega_peak = 1.6e-5 * (m_T / 1.22e19)**2  # assuming |beta|^2 ~ 1 at peak
    
    # Find best detector
    best_det = None
    best_gap = 1e100
    for name, det in detectors.items():
        f_lo, f_hi = det['f_range']
        if f_lo <= f_b <= f_hi:
            gap = det['Omega_h2'] / Omega_peak
            if gap < best_gap:
                best_gap = gap
                best_det = name
    
    if best_det is None:
        best_det = 'None'
        best_gap = float('inf')
    
    gap_str = f"{best_gap:.1e}" if best_gap < 1e50 else "No detector"
    print(f"{m_T:<12.1e} {f_b:<14.2e} {Omega_peak:<16.2e} {best_det:<16} {gap_str}")

## 9. Key Diagnostic: The (m_T/M_Pl)^2 Suppression

The central finding: the peak GW amplitude scales as $(m_T/M_{\rm Pl})^2$.
This is the standard vacuum amplification suppression, NOT the mass-coupling lock.
It applies to ANY bounce (or inflation).

In [ ]:
# What m_T is needed for detection?
M_Pl = 1.22e19  # GeV

for name, det in detectors.items():
    # Need: 1.6e-5 * (m_T/M_Pl)^2 >= Omega_sensitivity
    # m_T >= M_Pl * sqrt(Omega / 1.6e-5)
    m_T_min = M_Pl * np.sqrt(det['Omega_h2'] / 1.6e-5)
    f_b_at_min = 2.6e10 * np.sqrt(m_T_min / M_Pl)
    f_lo, f_hi = det['f_range']
    in_band = "YES" if f_lo <= f_b_at_min <= f_hi else "NO"
    print(f"{name:<12}: Need m_T > {m_T_min:.2e} GeV")
    print(f"             f_b = {f_b_at_min:.2e} Hz, in band: {in_band}")
    print()

## 10. Summary

### Expected results:

1. **|β_κ|² shape**: Approximately constant for κ < 1 (C₀ ~ 0.1–1), 
   oscillatory near κ ~ 1, exponentially suppressed for κ > 1.

2. **Peak amplitude**: Ω_GW h² ~ 1.6 × 10⁻⁵ × (m_T/M_Pl)² at f ~ f_b.

3. **Detectability**: Requires m_T > ~10¹⁶ GeV for ANY detector. 
   At this mass, f_b > 10⁸ Hz — far above all GW detector bands.

4. **The fundamental problem**: Lowering f_b into detector bands 
   requires lowering m_T, which lowers the amplitude as m_T².
   The amplitude drops FASTER than the frequency.